# Topic 30 · Sentiment of movie reviews: an LSTM against bag-of-words

CSCI 6379 · companion to the lecture note [Recurrent neural networks](https://dongchul.kim/csci6379-fall2026/topic/30).

**SST-2** (Stanford Sentiment Treebank): sentences from movie reviews, labelled positive or negative. An LSTM reads each sentence word by word and answers once at the end (**many-to-one**). We compare it with **bag-of-words**, which only counts which words appear and ignores their order.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes about 3 minutes on a T4.

In [ ]:
import urllib.request, zipfile, io, re, collections, time
import torch, torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

# SST-2 from the GLUE benchmark. Its own test labels are hidden, so the 872 dev sentences are our test set.
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://dl.fbaipublicfiles.com/glue/data/SST-2.zip").read()))
def read(name):
    rows = z.read(f"SST-2/{name}.tsv").decode("utf-8").strip().split("\n")[1:]
    return [r.split("\t")[0] for r in rows], [int(r.split("\t")[1]) for r in rows]
train_x, train_y = read("train"); test_x, test_y = read("dev")
print(len(train_x), "training sentences,", len(test_x), "test sentences")
for s, l in list(zip(train_x, train_y))[:5]:
    print(["negative", "positive"][l], "|", s)

## Baseline: bag-of-words

Count the words in each sentence and fit a logistic regression on the counts. Word order is thrown away.

In [ ]:
tok = lambda s: re.findall(r"[a-z']+|[.,!?;]", s.lower())
bow = CountVectorizer(tokenizer=tok, lowercase=False, token_pattern=None)
lr = LogisticRegression(max_iter=2000).fit(bow.fit_transform(train_x), train_y)
bow_acc = lr.score(bow.transform(test_x), test_y)
print(f"bag-of-words test accuracy: {bow_acc:.4f}")

## Words to ids, then the LSTM

Each word gets an id; an **embedding** turns the id into a vector the network learns.

In [ ]:
counts = collections.Counter(w for s in train_x for w in tok(s))
vocab = ["<pad>", "<unk>"] + [w for w, c in counts.items() if c >= 2]
wid = {w: i for i, w in enumerate(vocab)}
def encode(sents):
    ids = [[wid.get(w, 1) for w in tok(s)] or [1] for s in sents]
    L = max(len(i) for i in ids)
    return torch.tensor([i + [0] * (L - len(i)) for i in ids]), torch.tensor([len(i) for i in ids])
print(len(vocab), "words in the vocabulary")

class SentimentLSTM(nn.Module):
    def __init__(self, V, emb=100, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 2)

    def forward(self, x, lengths):
        # packing makes the LSTM stop at each sentence's real end instead of reading the padding
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(x), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.out(h[-1])                 # many-to-one: the memory after the last word

model = SentimentLSTM(len(vocab)).to(device)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
Xte, Lte = encode(test_x); Yte = torch.tensor(test_y)

def test_acc():
    model.eval()
    with torch.no_grad():
        acc = (model(Xte.to(device), Lte).argmax(1).cpu() == Yte).float().mean().item()
    model.train(); return acc

t0 = time.time()
for epoch in range(1, 5):
    order = torch.randperm(len(train_x))
    for i in range(0, len(order), 64):
        idx = order[i:i + 64]
        x, L = encode([train_x[j] for j in idx]); y = torch.tensor([train_y[j] for j in idx])
        loss = loss_fn(model(x.to(device), L), y.to(device))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch}  LSTM test accuracy {test_acc():.4f}  ({time.time() - t0:.0f}s)")
print(f"\nbag-of-words {bow_acc:.4f}  vs  LSTM {test_acc():.4f}")

## Does word order matter? Sentences with the same words in a different order

Bag-of-words sees exactly the same counts for each pair, so it **must** give both the same answer.

In [ ]:
def p_pos(s):
    model.eval()
    x, L = encode([s])
    with torch.no_grad():
        lstm = torch.softmax(model(x.to(device), L), 1)[0, 1].item()
    model.train()
    return lstm, lr.predict_proba(bow.transform([s]))[0, 1]

pairs = [("the movie was good , not bad .", "the movie was bad , not good ."),
         ("it is not good , it is bad .", "it is not bad , it is good ."),
         ("not boring , just funny .", "not funny , just boring .")]
print(f"{'sentence':40s} {'LSTM':>8s} {'bag-of-words':>13s}   (probability of positive)")
for a, b in pairs:
    for s in (a, b):
        l, w = p_pos(s)
        print(f"{s:40s} {l:8.2f} {w:13.2f}")
    print()